In [2]:
# Shared setup for final DAS classification visualizations
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REGISTRIES_DIR = PROJECT_ROOT / "data" / "metadata" / "registries"
SUBREGISTRIES_DIR = REGISTRIES_DIR / "output_subregistries"
OUTPUT_DIR = PROJECT_ROOT / "data" / "images" / "DAS" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_REGISTRY = REGISTRIES_DIR / "base_output_registry.csv"
DOCUMENT_REGISTRY = REGISTRIES_DIR / "document_registry.csv"
DAS_CLASSIFICATION_REGISTRY = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

LABEL_ORDER = [
    "no_data",
    "open_access",
    "partial_access",
    "restricted_access",
    "no_access",
    "incorrect",
    "unclear",
]
LABEL_DISPLAY = {
    "no_data": "No associated data",
    "open_access": "Open access",
    "partial_access": "Partial access",
    "restricted_access": "Restricted access",
    "no_access": "No access",
    "incorrect": "Incorrect / invalid",
    "unclear": "Unclear",
}
LABEL_COLORS = {
    "no_data": "#8E8E93",
    "open_access": "#1D9E75",
    "partial_access": "#66A61E",
    "restricted_access": "#E6A23C",
    "no_access": "#D95F02",
    "incorrect": "#7570B3",
    "unclear": "#6C757D",
}
PREFERRED_VERSIONS = ["v1.0.5", "v1.0.4", "v1.0.2", "v1.0.1", "v1.0.0", "v1.0.3"]


def load_resolved_das_classifications() -> pd.DataFrame:
    base = pd.read_csv(BASE_REGISTRY, dtype=str)
    documents = pd.read_csv(DOCUMENT_REGISTRY, dtype=str)
    das = pd.read_csv(DAS_CLASSIFICATION_REGISTRY, dtype=str)

    df = (
        das.merge(base, on="output_sha", how="inner")
        .query("output_type == 'classification'")
        .merge(
            documents[["doc_doi", "doc_type", "has_DAS", "publication_year", "journal"]],
            on="doc_doi",
            how="left",
        )
    )
    df = df[df["doc_type"].eq("Scientific study") & df["has_DAS"].eq("True")].copy()
    df["version_rank"] = df["software_version"].map({v: i for i, v in enumerate(PREFERRED_VERSIONS)}).fillna(999).astype(int)
    df["creation_date_sort"] = pd.to_datetime(df["creation_date"], errors="coerce")

    resolved = (
        df.sort_values(["doc_doi", "version_rank", "creation_date_sort"])
        .drop_duplicates("doc_doi", keep="first")
        .sort_values("doc_doi")
        .reset_index(drop=True)
    )
    assert not resolved["doc_doi"].duplicated().any(), "Final DAS table contains duplicate publications."
    assert resolved["output_sha"].is_unique, "Final DAS table contains duplicate classification output hashes."
    assert resolved["label"].isin(LABEL_ORDER).all(), "Unexpected DAS label encountered."
    return resolved


def ordered_counts(df: pd.DataFrame) -> pd.Series:
    return df["label"].value_counts().reindex(LABEL_ORDER, fill_value=0)


def save_label_bar(df: pd.DataFrame, title: str, outfile: str, subtitle: str | None = None) -> Path:
    counts = ordered_counts(df)
    total = int(counts.sum())
    fig_height = 4.8
    fig, ax = plt.subplots(figsize=(9.5, fig_height))
    y = range(len(counts))
    colors = [LABEL_COLORS[label] for label in counts.index]
    ax.barh(y, counts.values, color=colors, height=0.66)
    ax.set_yticks(list(y))
    ax.set_yticklabels([LABEL_DISPLAY[label] for label in counts.index])
    ax.invert_yaxis()
    ax.set_xlabel("Classified studies")
    fig.text(0.01, 0.985, title, ha="left", va="top", fontsize=13, fontweight="600")
    if subtitle:
        fig.text(0.01, 0.945, subtitle, ha="left", va="top", fontsize=9.5, color="#555")
    max_count = max(int(counts.max()), 1)
    ax.set_xlim(0, max_count * 1.18)
    for i, value in enumerate(counts.values):
        pct = (value / total * 100) if total else 0
        ax.text(value + max_count * 0.015, i, f"{int(value):,} ({pct:.1f}%)", va="center", fontsize=9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.subplots_adjust(top=0.82, left=0.24, right=0.98, bottom=0.12)
    path = OUTPUT_DIR / outfile
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def save_stacked_by_group(
    df: pd.DataFrame,
    group_col: str,
    title: str,
    outfile: str,
    *,
    exclude_no_data: bool = False,
    min_segment_pct_for_label: float = 8.5,
) -> Path:
    plot_df = df[df["label"].ne("no_data")].copy() if exclude_no_data else df.copy()
    labels = [label for label in LABEL_ORDER if (not exclude_no_data or label != "no_data")]
    group_order = plot_df[group_col].value_counts().index.tolist()
    counts = pd.crosstab(plot_df[group_col], plot_df["label"]).reindex(index=group_order).reindex(columns=labels, fill_value=0)
    totals = counts.sum(axis=1)
    pct = counts.div(totals.replace(0, pd.NA), axis=0).fillna(0) * 100

    fig_height = max(4.8, 0.58 * len(counts.index) + 1.7)
    fig, ax = plt.subplots(figsize=(11.5, fig_height))
    left = pd.Series(0.0, index=pct.index)
    segment_lefts = {}
    for label in labels:
        widths = pct[label]
        segment_lefts[label] = left.copy()
        ax.barh(pct.index, widths, left=left, color=LABEL_COLORS[label], label=LABEL_DISPLAY[label], height=0.68)
        left += widths

    for row_idx, group in enumerate(pct.index):
        row = pct.loc[group]
        labels_to_annotate = row[row >= min_segment_pct_for_label].sort_values(ascending=False).index
        for label in labels_to_annotate:
            width = row[label]
            x = segment_lefts[label].loc[group] + width / 2
            text_color = "white" if label in {"restricted_access", "no_access", "incorrect", "unclear"} else "#1F1F1F"
            label_fontsize = 7.6 if width < 8.5 else 8.5
            ax.text(x, group, f"{width:.1f}%", ha="center", va="center", fontsize=label_fontsize, color=text_color, fontweight="600")
    for group, total in totals.items():
        ax.text(101.2, group, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
    ax.set_xlim(0, 112)
    ax.set_xlabel("")
    fig.text(0.01, 0.985, title, ha="left", va="top", fontsize=13, fontweight="600")
    ax.invert_yaxis()
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=min(4, len(labels)), frameon=False)
    fig.subplots_adjust(top=0.90, left=0.28, right=0.94, bottom=0.22)
    path = OUTPUT_DIR / outfile
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


df_das_final = load_resolved_das_classifications()
print(f"Resolved DAS classification rows: {len(df_das_final):,}")
print(f"Unique publications: {df_das_final['doc_doi'].nunique():,}")
print(df_das_final["software_version"].value_counts().to_string())
print(df_das_final["publication_year"].value_counts().sort_index().to_string())


Resolved DAS classification rows: 10,391
Unique publications: 10,391
software_version
v1.0.4    10333
v1.0.5       58
publication_year
2024    2527
2025    7862
2026       2


In [ ]:
# Visualization: DAS classification output for all classified studies
path = save_label_bar(
    df_das_final,
    "DAS classification output for all classified studies",
    "DAS_classification_all_classified_studies.png",
    subtitle=f"Scientific studies with classified DAS sections; one resolved classification per publication (n={len(df_das_final):,}).",
)
print(path)


In [ ]:
# Visualization: DAS classification output for all classified studies with associated data
df_with_associated_data = df_das_final[df_das_final["label"].ne("no_data")].copy()
path = save_label_bar(
    df_with_associated_data,
    "DAS classification output for studies with associated data",
    "DAS_classification_associated_data_studies.png",
    subtitle=f"Excludes no_data classifications (n={len(df_with_associated_data):,} of {len(df_das_final):,}).",
)
print(path)


In [3]:
# Visualization: DAS classification output by year, excluding 2026
df_2024_2025 = df_das_final[df_das_final["publication_year"].isin(["2024", "2025"])].copy()
path = save_stacked_by_group(
    df_2024_2025,
    "publication_year",
    "DAS classification output by publication year",
    "DAS_classification_by_year_excluding_2026.png",
    min_segment_pct_for_label=6.8,
)
print(path)


/home/jzupp/projects/medialab_internship/data/images/DAS/final/DAS_classification_by_year_excluding_2026.png


In [ ]:
# Visualization: DAS classification output for 2024 studies
df_2024 = df_das_final[df_das_final["publication_year"].eq("2024")].copy()
path = save_label_bar(
    df_2024,
    "DAS classification output for 2024 studies",
    "DAS_classification_2024_studies.png",
    subtitle=f"Scientific studies with classified DAS sections published in 2024 (n={len(df_2024):,}).",
)
print(path)


In [ ]:
# Visualization: DAS classification output for 2025 studies
df_2025 = df_das_final[df_das_final["publication_year"].eq("2025")].copy()
path = save_label_bar(
    df_2025,
    "DAS classification output for 2025 studies",
    "DAS_classification_2025_studies.png",
    subtitle=f"Scientific studies with classified DAS sections published in 2025 (n={len(df_2025):,}).",
)
print(path)


In [ ]:
# Visualization: DAS classification output by journal for all classified studies
path = save_stacked_by_group(
    df_das_final.assign(journal=df_das_final["journal"].fillna("Unknown journal")),
    "journal",
    "DAS classification output by journal for all classified studies",
    "DAS_classification_by_journal_all_classified_studies.png",
)
print(path)


In [ ]:
# Visualization: DAS classification output by journal for 2024 studies
df_2024 = df_das_final[df_das_final["publication_year"].eq("2024")].copy()
path = save_stacked_by_group(
    df_2024.assign(journal=df_2024["journal"].fillna("Unknown journal")),
    "journal",
    "DAS classification output by journal for 2024 studies",
    "DAS_classification_by_journal_2024_studies.png",
)
print(path)


In [ ]:
# Visualization: DAS classification output by journal for 2025 studies
df_2025 = df_das_final[df_das_final["publication_year"].eq("2025")].copy()
path = save_stacked_by_group(
    df_2025.assign(journal=df_2025["journal"].fillna("Unknown journal")),
    "journal",
    "DAS classification output by journal for 2025 studies",
    "DAS_classification_by_journal_2025_studies.png",
)
print(path)


In [ ]:
# Visualization: DAS classification output by journal for studies with associated data
df_with_associated_data = df_das_final[df_das_final["label"].ne("no_data")].copy()
path = save_stacked_by_group(
    df_with_associated_data.assign(journal=df_with_associated_data["journal"].fillna("Unknown journal")),
    "journal",
    "DAS classification output by journal for studies with associated data",
    "DAS_classification_by_journal_associated_data_studies.png",
    exclude_no_data=True,
)
print(path)
